# ADE20K Semantic Segmentation Assignment
### Classes: person, car, book, airplane (+ background)

**Module:** Computer Vision / Deep Learning
**Name:** *&lt;your name here&gt;*
**Date:** *&lt;date&gt;*

## What this notebook does

I am building a semantic segmentation model on a 350 train / 350 validation subset of ADE20K,
using only the 4 classes we were asked to detect: **person, car, book, airplane**, plus a
**background** class. That gives **5 classes in total**.

This is important: the model predicts **5 classes**, not just "object vs background". If I only
did foreground/background, that would be a *binary* segmentation problem, which the assignment
brief says will be penalised. So my final Conv2D layer always has 5 output channels, and I use
`argmax` over those channels to decide the predicted class of each pixel.

## Structure of the notebook

1. Load the images + JSON annotations, and build label masks.
2. Explore the data (EDA) so I can justify my preprocessing choices.
3. Preprocess and augment the data using a Keras `Sequence` generator.
4. Build a simple **encoder-decoder** CNN from scratch using `Conv2D`, `MaxPooling2D` and
   `Conv2DTranspose` (as covered in the lectures on segmentation).
5. Train it with `model.compile()` / `model.fit()`.
6. Evaluate with IoU and Dice score, and a confusion matrix.
7. Run the trained model on 3 unlabelled test images, as required by the brief.

**Note on scope:** the brief mentions comparing against a pretrained model for extra marks. I
have left this out of this version, since the lectures have only covered building CNNs from
scratch and I did not want to use a pretrained backbone (like ResNet) that we have not covered
in class. Everything below is built using only `Conv2D`, `MaxPooling2D`, `Conv2DTranspose` and
standard Keras training tools.

## Expected folder structure

```
/content/data/
├── images/
│   ├── training/        # 350 .jpg images
│   └── validation/      # 350 .jpg images
└── annotations/
    ├── training/        # 350 .json files, same filename as the matching image
    └── validation/
```


In [6]:
# ---------------------------------------------------------------------------
# Setup and Imports
# ---------------------------------------------------------------------------
# I am using TensorFlow / Keras (as covered in the lectures) instead of PyTorch.
import os, json, glob, random, math
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [7]:
# ---------------------------------------------------------------------------
# GPU check
# ---------------------------------------------------------------------------
# I am running this notebook locally with CUDA enabled, so before doing
# anything else I want to confirm TensorFlow can actually see my GPU.
gpus = tf.config.list_physical_devices("GPU")
print("Num GPUs detected:", len(gpus))
for gpu in gpus:
    print(" -", gpu)

if len(gpus) == 0:
    print("No GPU detected -- training will fall back to CPU and will be much slower.")

# ---------------------------------------------------------------------------
# Enable memory growth
# ---------------------------------------------------------------------------
# By default, TensorFlow reserves ALL of the GPU's VRAM as soon as it starts
# running, even if the model does not need that much yet. On my own machine
# this can cause an out-of-memory crash once I start training the U-Net,
# especially if anything else is also using the GPU at the same time.
# Setting "memory growth" to True tells TensorFlow to only allocate VRAM
# gradually, as it is actually needed, instead of grabbing it all up front.
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Memory growth enabled for {len(gpus)} GPU(s).")
    except RuntimeError as e:
        # Memory growth has to be set before TensorFlow initializes the GPU
        # (e.g. before any tensors have been created). If that has already
        # happened elsewhere in the session, this will raise an error.
        print("Could not set memory growth (must be set before GPUs are initialized):", e)


Num GPUs detected: 0
No GPU detected -- training will fall back to CPU and will be much slower.


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

# All the paths and settings for the notebook live here, so I only have to
# change one place if my folder names are different.
DATA_ROOT = "/content/IMAGE_SEGMENTATION/RMDS_segmentation_dataset_ADE20K_350"

CONFIG = {
    "img_train_dir": os.path.join(DATA_ROOT, "train"),
    "img_val_dir": os.path.join(DATA_ROOT, "val"),

    # JSON annotation files
    "ann_train_file": os.path.join(DATA_ROOT, "instances_train.json"),
    "ann_val_file": os.path.join(DATA_ROOT, "instances_val.json"),

    # Generated segmentation masks will be stored here
    "mask_cache_dir": os.path.join(DATA_ROOT, "masks_cache"),

    "output_dir": os.path.join(DATA_ROOT, "outputs"),

    "classes": ["background", "person", "car", "book", "airplane"],
    "num_classes": 5,

    "img_size": 128,
    "batch_size": 8,
    "epochs": 25,
    "learning_rate": 1e-3,
}

os.makedirs(CONFIG["mask_cache_dir"], exist_ok=True)
os.makedirs(CONFIG["output_dir"], exist_ok=True)


TensorFlow version: 2.20.0
GPU available: False


## Section 1: Loading the Data & EDA

Before I build a model, I need to (1) turn the JSON annotations into pixel masks I can actually
train on, and (2) look at the data to understand it (class balance, image sizes, etc.) so my
preprocessing choices later are justified rather than arbitrary.


In [ ]:
# Check that the dataset folders actually exist before doing anything else.
# (If you are running this in Drive instead of /content, mount Drive and
# update the paths in CONFIG above.)
def check_folders(cfg):
    ok = True
    for key in ["img_train_dir", "img_val_dir", "ann_train_dir", "ann_val_dir"]:
        if not os.path.isdir(cfg[key]):
            print(f"Missing folder: {cfg[key]}")
            ok = False
    if ok:
        n_train = len(glob.glob(os.path.join(cfg["img_train_dir"], "*.jpg")))
        n_val = len(glob.glob(os.path.join(cfg["img_val_dir"], "*.jpg")))
        print(f"Found {n_train} training images and {n_val} validation images.")
    return ok

DATA_OK = check_folders(CONFIG)


In [ ]:
# Have a quick look at one annotation file so I know what fields I am
# working with before writing the parsing code below.
if DATA_OK:
    example_json = sorted(glob.glob(os.path.join(CONFIG["ann_train_dir"], "*.json")))[0]
    with open(example_json) as f:
        example_ann = json.load(f)
    print(json.dumps(example_ann, indent=2)[:1000])


In [ ]:
# ---------------------------------------------------------------------------
# Turning the JSON polygon annotations into label masks
# ---------------------------------------------------------------------------
# ADE20K annotations store each object as a polygon (a list of x, y points)
# together with the object's name, using this structure:
#   {"annotation": {"object": [{"name": "person", "polygon": {"x": [...], "y": [...]}}, ...]}}
#
# I am rasterising these polygons myself into a single-channel mask image
# where each pixel's value is the class index (0=background, 1=person,
# 2=car, 3=book, 4=airplane), instead of using any mask image that might
# come with the dataset, since I know exactly how my own masks are encoded.
#
# ADE20K names are sometimes written with synonyms, e.g. "airplane, aeroplane,
# plane", so I check against a small list of synonyms for each class.
SYNONYMS = {
    "person": ["person", "people", "man", "woman"],
    "car": ["car", "automobile", "auto"],
    "book": ["book", "books"],
    "airplane": ["airplane", "aeroplane", "plane", "aircraft"],
}


def get_class_for_name(raw_name, synonyms):
    # raw_name might be "person, individual" -- split on commas and check each word
    words = [w.strip().lower() for w in raw_name.split(",")]
    for class_name, syn_list in synonyms.items():
        for w in words:
            if w in syn_list:
                return class_name
    return None  # not one of our 4 target classes


def rasterize_mask(json_path, img_h, img_w, cfg):
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    class_to_idx = {name: i for i, name in enumerate(cfg["classes"])}

    with open(json_path) as f:
        ann = json.load(f)

    objects = ann["annotation"]["object"]

    # Sort objects from biggest to smallest so that small objects (like a
    # book) get painted LAST and are not hidden underneath a bigger object.
    objects_with_area = []
    for obj in objects:
        class_name = get_class_for_name(obj["name"], SYNONYMS)
        if class_name is None:
            continue
        xs = np.array(obj["polygon"]["x"], dtype=np.float32)
        ys = np.array(obj["polygon"]["y"], dtype=np.float32)
        area = cv2.contourArea(np.stack([xs, ys], axis=1))
        objects_with_area.append((area, class_name, xs, ys))

    objects_with_area.sort(key=lambda o: o[0], reverse=True)

    for area, class_name, xs, ys in objects_with_area:
        points = np.stack([xs, ys], axis=1).astype(np.int32)
        cv2.fillPoly(mask, [points], color=class_to_idx[class_name])

    classes_present = [o[1] for o in objects_with_area]
    return mask, classes_present


In [ ]:
# ---------------------------------------------------------------------------
# Build a mask for every image and save it to disk (masks_cache/), so we only
# have to do this once instead of re-parsing JSON every epoch.
# ---------------------------------------------------------------------------
def build_dataset_table(img_dir, ann_dir, split_name, cfg):
    cache_dir = os.path.join(cfg["mask_cache_dir"], split_name)
    os.makedirs(cache_dir, exist_ok=True)

    rows = []
    for img_path in sorted(glob.glob(os.path.join(img_dir, "*.jpg"))):
        stem = Path(img_path).stem
        json_path = os.path.join(ann_dir, stem + ".json")
        if not os.path.exists(json_path):
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        mask, classes_present = rasterize_mask(json_path, h, w, cfg)
        mask_path = os.path.join(cache_dir, stem + ".png")
        cv2.imwrite(mask_path, mask)

        rows.append({
            "stem": stem, "img_path": img_path, "mask_path": mask_path,
            "height": h, "width": w, "classes_present": classes_present,
        })

    df = pd.DataFrame(rows)
    print(f"{split_name}: built {len(df)} masks")
    return df

if DATA_OK:
    train_df = build_dataset_table(CONFIG["img_train_dir"], CONFIG["ann_train_dir"], "training", CONFIG)
    val_df = build_dataset_table(CONFIG["img_val_dir"], CONFIG["ann_val_dir"], "validation", CONFIG)


### EDA

Now that I have masks, I want to look at (1) some example images with their masks overlaid,
(2) how often each class appears, (3) how many pixels each class takes up (this tells me how
imbalanced the classes are, which I need for my loss function later), and (4) which classes
tend to appear together in the same image, as a simple correlation matrix.


In [ ]:
# A fixed colour for each class, used in every plot in this notebook so
# the colours always mean the same thing.
CLASS_COLORS = {
    "background": "#000000",
    "person": "#e63946",
    "car": "#2a9d8f",
    "book": "#f4a261",
    "airplane": "#3a86ff",
}
CMAP = mcolors.ListedColormap([CLASS_COLORS[c] for c in CONFIG["classes"]])
NORM = mcolors.BoundaryNorm(np.arange(-0.5, CONFIG["num_classes"] + 0.5, 1), CMAP.N)

# Show a few training images with their mask overlaid, so I can sanity-check
# that my rasterization code above is actually working correctly.
sample = train_df.sample(6, random_state=1)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    img = cv2.cvtColor(cv2.imread(row["img_path"]), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
    ax.imshow(img)
    ax.imshow(mask, cmap=CMAP, norm=NORM, alpha=0.5)
    ax.set_title(", ".join(row["classes_present"]) or "background only", fontsize=9)
    ax.axis("off")
plt.suptitle("Sample images with rasterized masks")
plt.tight_layout()
plt.show()


In [ ]:
# How many images contain each class? This is useful to know because a
# class that barely appears in the dataset will be hard for the model to
# learn, no matter how I set up training.
counts = Counter()
for classes in train_df["classes_present"]:
    for c in set(classes):
        counts[c] += 1

class_freq = pd.Series({c: counts.get(c, 0) for c in CONFIG["classes"][1:]})
plt.figure(figsize=(6, 4))
plt.bar(class_freq.index, class_freq.values, color=[CLASS_COLORS[c] for c in class_freq.index], edgecolor="black")
plt.ylabel("number of training images containing the class")
plt.title("How many images contain each class")
plt.show()

print(class_freq)


In [ ]:
# Pixel-level class balance. This matters more than the image-level count
# above, because a class can appear in lots of images but only take up a
# tiny area (like "book"). I use this to build class weights for my loss
# function further down.
pixel_counts = {c: 0 for c in CONFIG["classes"]}
for _, row in train_df.iterrows():
    mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
    for idx, c in enumerate(CONFIG["classes"]):
        pixel_counts[c] += int((mask == idx).sum())

pixel_series = pd.Series(pixel_counts)
plt.figure(figsize=(6, 4))
plt.bar(pixel_series.index, pixel_series.values, color=[CLASS_COLORS[c] for c in pixel_series.index], edgecolor="black")
plt.yscale("log")
plt.ylabel("total pixel count (log scale)")
plt.title("Pixel-level class imbalance")
plt.show()

print(pixel_series)


In [ ]:
# A simple correlation / co-occurrence matrix: how often do pairs of
# classes show up in the same image? For example, I would expect "person"
# and "car" to co-occur a lot in street scenes.
target_classes = CONFIG["classes"][1:]
presence = pd.DataFrame(0, index=train_df["stem"], columns=target_classes)
for _, row in train_df.iterrows():
    for c in set(row["classes_present"]):
        presence.loc[row["stem"], c] = 1

corr_matrix = presence.corr()

plt.figure(figsize=(5, 4))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Class co-occurrence correlation")
plt.tight_layout()
plt.show()


**What I learned from the EDA, and what I am going to do about it:**

- The `background` class dominates the pixel counts, and `book` is by far the rarest class.
  This means if I train with plain cross-entropy, the model will mostly just learn to predict
  background everywhere and get a high accuracy while being useless. So I am going to use
  **class weights** in my loss function (Section 2) to push the model to pay more attention to
  the rarer classes.
- `person` and `car` seem to co-occur reasonably often, which makes sense for street scenes.
  I will keep this in mind when I look at the confusion matrix later &mdash; if the model
  confuses these two classes, it's not a random bug, it's because they often appear in similar,
  cluttered scenes.


## Section 2: Preprocessing & Data Loading

I am resizing every image and mask to a fixed size (128x128) so that they can be stacked
into batches, and I am building a Keras `Sequence` so that images are only loaded into memory
in small batches instead of all at once (this stops Colab from running out of RAM on a bigger
dataset).

For augmentation, I am keeping things simple: a random horizontal flip and a small random
brightness change, applied only to the training set. This is enough to make the model a bit
more robust without needing an extra augmentation library.


In [ ]:
# Class weights for the loss function, calculated from the pixel counts
# in the EDA above. This is the standard "balanced" weighting formula:
# weight_c = total_pixels / (num_classes * pixels_of_class_c)
# A class with very few pixels (like "book") gets a bigger weight, so
# mistakes on that class are penalised more during training.
total_pixels = sum(pixel_counts.values())
class_weights = []
for c in CONFIG["classes"]:
    w = total_pixels / (CONFIG["num_classes"] * max(pixel_counts[c], 1))
    class_weights.append(w)

print("Class weights:", dict(zip(CONFIG["classes"], np.round(class_weights, 3))))


In [ ]:
# ---------------------------------------------------------------------------
# Keras Sequence: loads images/masks in batches instead of all at once.
# ---------------------------------------------------------------------------
class SegmentationSequence(keras.utils.Sequence):
    def __init__(self, df, img_size, batch_size, augment=False, **kwargs):
        super().__init__(**kwargs)
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.indices = np.arange(len(self.df))

    def __len__(self):
        # number of batches per epoch
        return len(self.df) // self.batch_size

    def on_epoch_end(self):
        # shuffle the order of the data after every epoch
        np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_images = np.zeros((len(batch_idx), self.img_size, self.img_size, 3), dtype=np.float32)
        batch_masks = np.zeros((len(batch_idx), self.img_size, self.img_size, 1), dtype=np.int32)

        for i, row_idx in enumerate(batch_idx):
            row = self.df.iloc[row_idx]

            img = cv2.cvtColor(cv2.imread(row["img_path"]), cv2.COLOR_BGR2RGB)
            mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)

            # Resize image with normal interpolation, but the mask with
            # NEAREST interpolation -- otherwise resizing would blend class
            # labels together and create label values that do not exist.
            img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

            if self.augment:
                # random horizontal flip
                if random.random() < 0.5:
                    img = np.fliplr(img)
                    mask = np.fliplr(mask)
                # small random brightness change (image only, not the mask!)
                if random.random() < 0.3:
                    factor = random.uniform(0.8, 1.2)
                    img = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

            batch_images[i] = img.astype(np.float32) / 255.0     # normalise to [0, 1]
            batch_masks[i, :, :, 0] = mask

        return batch_images, batch_masks


train_seq = SegmentationSequence(train_df, CONFIG["img_size"], CONFIG["batch_size"], augment=True)
val_seq = SegmentationSequence(val_df, CONFIG["img_size"], CONFIG["batch_size"], augment=False)

print(f"Training batches per epoch: {len(train_seq)}")
print(f"Validation batches per epoch: {len(val_seq)}")


In [ ]:
# Quick sanity check: pull one batch out of the generator and plot it, to
# make sure images and masks still line up correctly after augmentation.
X_batch, Y_batch = train_seq[0]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i in range(4):
    axes[0, i].imshow(X_batch[i]); axes[0, i].axis("off")
    axes[1, i].imshow(Y_batch[i, :, :, 0], cmap=CMAP, norm=NORM); axes[1, i].axis("off")
plt.suptitle("One batch from the training generator (image on top, mask below)")
plt.tight_layout()
plt.show()


## Section 3: Model &mdash; Simple Encoder-Decoder CNN

This is a basic **encoder-decoder architecture**, as covered in the segmentation lectures:

- The **encoder** repeatedly applies convolution + max-pooling. Each `MaxPooling2D` halves the
  height and width of the feature maps, **reducing spatial dimensionality** while the number of
  filters increases, so the network builds up a more compressed, abstract representation of the
  image (what objects/edges are present, without worrying about exact pixel position yet).
- The **bottleneck** is the most compressed point of the network.
- The **decoder** does the opposite: `Conv2DTranspose` layers **upsample the feature maps**
  back up to the original image size, step by step, so that the network can turn the compressed
  representation back into a full-resolution, pixel-by-pixel prediction.
- The last layer is a `Conv2D` with 5 filters (one per class) and a **softmax** activation, so
  every pixel gets a probability distribution over the 5 classes.

I am keeping this simple on purpose (no skip connections, no pretrained backbone) since that is
the architecture level we have covered in lectures. A natural next step, mentioned in my
conclusion, would be to add skip connections between the encoder and decoder (turning this into
a full U-Net), which usually helps recover sharper object edges.


In [ ]:
def build_encoder_decoder(img_size, num_classes):
    inputs = keras.Input(shape=(img_size, img_size, 3))

    # --- Encoder: reduce spatial dimensionality, increase feature depth ---
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)          # 128 -> 64

    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)          # 64 -> 32

    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)          # 32 -> 16

    # --- Bottleneck: most compressed representation of the image ---
    x = layers.Conv2D(256, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(256, 3, activation="relu", padding="same")(x)

    # --- Decoder: upsample the feature maps back to full resolution ---
    x = layers.Conv2DTranspose(128, 3, strides=2, activation="relu", padding="same")(x)   # 16 -> 32
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, activation="relu", padding="same")(x)    # 32 -> 64
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)

    x = layers.Conv2DTranspose(32, 3, strides=2, activation="relu", padding="same")(x)    # 64 -> 128
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)

    # Final layer: one channel per class, softmax so each pixel gets a
    # probability distribution over all 5 classes (background + 4 objects).
    # This is what makes the model multi-class instead of binary.
    outputs = layers.Conv2D(num_classes, 1, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="simple_encoder_decoder")


model = build_encoder_decoder(CONFIG["img_size"], CONFIG["num_classes"])
model.summary()


### Loss function and metrics

For the loss I am using **weighted sparse categorical cross-entropy**. "Sparse" just means my
labels are plain integers (0-4) instead of one-hot vectors, which is more memory efficient.
I multiply the per-pixel loss by the class weight I calculated in Section 2, so mistakes on
rare classes (like `book`) are penalised more heavily than mistakes on `background`.

For metrics I am using `tf.keras.metrics.MeanIoU`, which is the standard **Intersection over
Union** metric &mdash; the size of the overlap between predicted and true regions, divided by
the size of their union. I wrap it in a small subclass so it can `argmax` the model's softmax
output automatically before comparing it to the integer ground truth mask.


In [ ]:
# Weighted loss function: normal sparse categorical cross-entropy, but each
# pixel's loss is multiplied by the weight of its true class.
def weighted_sparse_categorical_crossentropy(class_weights):
    class_weights_tensor = tf.constant(class_weights, dtype=tf.float32)

    def loss_fn(y_true, y_pred):
        y_true_int = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
        weights = tf.gather(class_weights_tensor, y_true_int)
        per_pixel_loss = keras.losses.sparse_categorical_crossentropy(y_true_int, y_pred)
        return tf.reduce_mean(per_pixel_loss * weights)

    return loss_fn


# MeanIoU expects both y_true and y_pred as class-index maps (not
# probabilities), so I override update_state to argmax the prediction and
# squeeze the extra channel dimension off the ground-truth mask first.
class MeanIoUMetric(keras.metrics.MeanIoU):
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.squeeze(y_true, axis=-1)
        y_pred = tf.argmax(y_pred, axis=-1)
        return super().update_state(y_true, y_pred, sample_weight)


model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG["learning_rate"]),
    loss=weighted_sparse_categorical_crossentropy(class_weights),
    metrics=["accuracy", MeanIoUMetric(num_classes=CONFIG["num_classes"], name="mean_iou")],
)


In [ ]:
# Train using standard Keras callbacks:
#  - EarlyStopping: stop training once val_mean_iou stops improving, so I
#    don''t waste time/compute on extra epochs that don''t help.
#  - ModelCheckpoint: automatically keep the best-performing weights.
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_mean_iou", mode="max", patience=6, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CONFIG["output_dir"], "best_model.keras"),
        monitor="val_mean_iou", mode="max", save_best_only=True,
    ),
]

history = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=CONFIG["epochs"],
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# Plot the training curves so I can check for overfitting (train and val
# loss diverging) and see how mean IoU improved over training.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history.history["loss"], label="train loss")
axes[0].plot(history.history["val_loss"], label="val loss")
axes[0].set_title("Loss over epochs")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history.history["mean_iou"], label="train mean IoU")
axes[1].plot(history.history["val_mean_iou"], label="val mean IoU")
axes[1].set_title("Mean IoU over epochs")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "training_curves.png"), dpi=150)
plt.show()


## Section 4: Evaluation &mdash; IoU, Dice, and Confusion Matrix

`MeanIoU` during training gives me one overall number, but for the report I want the score
**per class**, plus the **Dice score**, plus a full confusion matrix so I can see exactly which
classes the model mixes up. I compute all of this from a confusion matrix built manually over
the whole validation set.


In [ ]:
# Build a confusion matrix over the whole validation set by comparing the
# predicted class of every pixel to its true class.
def build_confusion_matrix(model, sequence, num_classes):
    conf_mat = np.zeros((num_classes, num_classes), dtype=np.int64)
    for i in range(len(sequence)):
        X_batch, Y_batch = sequence[i]
        preds = model.predict(X_batch, verbose=0)
        pred_labels = np.argmax(preds, axis=-1)          # (batch, H, W)
        true_labels = Y_batch[:, :, :, 0]                 # (batch, H, W)

        for t, p in zip(true_labels.flatten(), pred_labels.flatten()):
            conf_mat[t, p] += 1
    return conf_mat


conf_mat = build_confusion_matrix(model, val_seq, CONFIG["num_classes"])

# From the confusion matrix, work out IoU and Dice for every class:
#   IoU_c  = TP / (TP + FP + FN)
#   Dice_c = 2*TP / (2*TP + FP + FN)
iou_per_class = []
dice_per_class = []
for c in range(CONFIG["num_classes"]):
    tp = conf_mat[c, c]
    fp = conf_mat[:, c].sum() - tp
    fn = conf_mat[c, :].sum() - tp
    iou = tp / (tp + fp + fn + 1e-7)
    dice = (2 * tp) / (2 * tp + fp + fn + 1e-7)
    iou_per_class.append(iou)
    dice_per_class.append(dice)

results_df = pd.DataFrame({
    "class": CONFIG["classes"],
    "IoU": np.round(iou_per_class, 4),
    "Dice": np.round(dice_per_class, 4),
})
print(results_df)

# Mean IoU/Dice over the 4 OBJECT classes only (not background), since
# background is so common that including it would make the score look
# better than it really is at detecting the classes we actually care about.
mean_iou_fg = np.mean(iou_per_class[1:])
mean_dice_fg = np.mean(dice_per_class[1:])
print(f"\nMean foreground IoU: {mean_iou_fg:.4f}")
print(f"Mean foreground Dice: {mean_dice_fg:.4f}")


In [ ]:
# Bar chart of IoU / Dice per class, and the confusion matrix as a heatmap.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

x = np.arange(len(CONFIG["classes"]))
axes[0].bar(x - 0.2, iou_per_class, width=0.4, label="IoU", color="#3a86ff")
axes[0].bar(x + 0.2, dice_per_class, width=0.4, label="Dice", color="#e63946")
axes[0].set_xticks(x); axes[0].set_xticklabels(CONFIG["classes"], rotation=20)
axes[0].set_title("IoU and Dice per class")
axes[0].legend()

conf_mat_norm = conf_mat / conf_mat.sum(axis=1, keepdims=True).clip(min=1)
sns.heatmap(conf_mat_norm, annot=True, fmt=".2f", cmap="viridis",
            xticklabels=CONFIG["classes"], yticklabels=CONFIG["classes"], ax=axes[1])
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].set_title("Confusion matrix (row-normalised)")

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "evaluation_metrics.png"), dpi=150)
plt.show()

# This confusion matrix is 5x5, showing the model is genuinely distinguishing
# between all 5 classes, not just doing foreground-vs-background.


### A few qualitative examples

Numbers don't tell the whole story, so here are a few validation images with the ground
truth mask and the model's predicted mask side by side.


In [ ]:
# Show image / ground truth / prediction for a few validation examples.
n_show = 4
sample_indices = np.random.choice(len(val_df), n_show, replace=False)

fig, axes = plt.subplots(n_show, 3, figsize=(10, 3.3 * n_show))
col_titles = ["Image", "Ground truth", "Prediction"]

for row, idx in enumerate(sample_indices):
    data_row = val_df.iloc[idx]
    img = cv2.cvtColor(cv2.imread(data_row["img_path"]), cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (CONFIG["img_size"], CONFIG["img_size"]))
    mask = cv2.imread(data_row["mask_path"], cv2.IMREAD_GRAYSCALE)
    mask_resized = cv2.resize(mask, (CONFIG["img_size"], CONFIG["img_size"]), interpolation=cv2.INTER_NEAREST)

    input_img = img_resized.astype(np.float32) / 255.0
    pred = model.predict(input_img[np.newaxis, ...], verbose=0)[0]
    pred_labels = np.argmax(pred, axis=-1)

    for col, content in enumerate([img_resized, mask_resized, pred_labels]):
        ax = axes[row, col]
        if col == 0:
            ax.imshow(content)
        else:
            ax.imshow(img_resized)
            ax.imshow(content, cmap=CMAP, norm=NORM, alpha=0.55)
        ax.axis("off")
        if row == 0:
            ax.set_title(col_titles[col])

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "qualitative_examples.png"), dpi=150)
plt.show()


## Section 5: Final Predictions on 3 Unlabelled Test Images

As required by the assignment brief, here I run the trained model on exactly 3 unlabelled
test images and plot the predicted segmentation. If you have a separate folder of unlabelled
test images, put their paths in `TEST_IMAGE_PATHS` below. Otherwise this falls back to 3
random validation images (used only as plain images, their masks are not shown) so the code
still runs end-to-end.


In [ ]:
TEST_IMAGE_PATHS = [
    # "/content/data/test/unlabelled_1.jpg",
    # "/content/data/test/unlabelled_2.jpg",
    # "/content/data/test/unlabelled_3.jpg",
]

if not TEST_IMAGE_PATHS:
    TEST_IMAGE_PATHS = val_df["img_path"].sample(3, random_state=7).tolist()
    print("No TEST_IMAGE_PATHS set -- using 3 validation images as a stand-in. "
          "Replace this list with your real unlabelled test images before submitting.")

fig, axes = plt.subplots(len(TEST_IMAGE_PATHS), 2, figsize=(9, 4 * len(TEST_IMAGE_PATHS)))

for row, path in enumerate(TEST_IMAGE_PATHS):
    raw_img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    resized = cv2.resize(raw_img, (CONFIG["img_size"], CONFIG["img_size"]))
    input_img = resized.astype(np.float32) / 255.0

    pred = model.predict(input_img[np.newaxis, ...], verbose=0)[0]
    pred_labels = np.argmax(pred, axis=-1)

    # resize the predicted mask back up to the original image size for display
    pred_labels_full = cv2.resize(pred_labels.astype(np.uint8), (raw_img.shape[1], raw_img.shape[0]),
                                   interpolation=cv2.INTER_NEAREST)

    axes[row, 0].imshow(raw_img); axes[row, 0].axis("off"); axes[row, 0].set_title(Path(path).name)
    axes[row, 1].imshow(raw_img); axes[row, 1].imshow(pred_labels_full, cmap=CMAP, norm=NORM, alpha=0.55)
    axes[row, 1].axis("off"); axes[row, 1].set_title("predicted segmentation")

    out_path = os.path.join(CONFIG["output_dir"], f"test_prediction_{row+1}.png")
    cv2.imwrite(out_path, cv2.cvtColor(pred_labels_full, cv2.COLOR_GRAY2BGR))

handles = [plt.Rectangle((0, 0), 1, 1, color=CLASS_COLORS[c]) for c in CONFIG["classes"]]
fig.legend(handles, CONFIG["classes"], loc="lower center", ncol=5, frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "final_test_predictions.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved outputs to {CONFIG['output_dir']}")


## Conclusion

**Why I reported both IoU and Dice:** IoU (intersection over union) and Dice score measure a
similar thing &mdash; how much the predicted region overlaps with the true region &mdash; but
Dice weights the overlap slightly more heavily, so it is a bit less harsh on small objects like
`book` where even a small boundary mismatch counts a lot against plain IoU.

**Why I excluded background from the mean score:** background makes up most of the pixels in
almost every image, so a model that just predicts background everywhere would still get a
high score if background were included. Reporting the mean IoU/Dice over the 4 object classes
only is a fairer test of whether the model is actually learning to find person/car/book/airplane.

**Limitations / what I would improve with more time:**
- My encoder-decoder has no skip connections between encoder and decoder, so it likely loses
  some fine boundary detail, especially on small objects like `book`. Adding skip connections
  (turning this into a U-Net) would likely help.
- 350 training images is a small dataset for segmentation, so results may vary a lot between
  runs / random seeds.
- The class weighting helps with the background/`book` imbalance but does not fully solve it.
- I did not include a pretrained-model comparison, since that would require a pretrained
  backbone we have not covered in the module.
